In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from IPython.display import clear_output, display
from tqdm.auto import tqdm
from argparse import Namespace

from ncpu.dataset import NCPUDataset, sample_4bit_adder
from ncpu.nca import NeuralCA
from ncpu.trainer import NCPUTrainer

In [ ]:
device = "cuda"
ds_config = Namespace(
    W=64,
    H=64,
    r=3,
    spacing=(1, 29),
    sampler=sample_4bit_adder,
    balanced=False,
)

nca_config = Namespace(
    channels=16,
    hidden_channels=[100],
    fire_rate=0.9,
    alive_threshold=0.1,
    zero_initialization=True,
    kernel_size=3,
    num_perception_kernels=3,
    read_only_dims=[-1, -2],
)

optim_config = Namespace(
    lr=0.00001,
    batch_size=12,
    gaussian_noise=-1,
)

In [ ]:
dataset = NCPUDataset(ds_config)
nca = NeuralCA(**vars(nca_config)).to(device)
new_trainer = NCPUTrainer(
    nca,
    dataset.get_dataloader(batch_size=optim_config.batch_size),
    lr=optim_config.lr,
    gaussian_noise=optim_config.gaussian_noise,
)

# trainer = NCPUTrainer.load_last_trainer()
# trainer.load_checkpoint(41860)
# trainer.ds = new_trainer.ds
# trainer.dataloader = new_trainer.dataloader
# trainer.dataset_iter = new_trainer.dataset_iter
trainer = new_trainer

trainer.sanity_check()

In [ ]:
with torch.no_grad():
    info = trainer.optim_step(steps=(100, 101))
    trainer.display_optim_step(info, display_size=117, to_show=8)

In [ ]:
pbar = tqdm(range(1_000_000))

for i in pbar:
    info = trainer.optim_step(steps=(10, 51))
    loss = info["loss"]
    pbar.set_description(f"loss={loss:.6f}")

    if i % 250 == 0:
        clear_output(wait=False)
        display(pbar.container)

        trainer.display_optim_step(info, to_show=8)
        trainer.save_checkpoint()

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import panel as pn
pn.extension()
import mediapy as media
from ncpu.utils import tensor_to_video_pane

# Example video tensor: 4 videos, 30 frames, 3 channels, 64x64
B, T, C, H, W = 4, 30, 3, 32, 32
videos = torch.randn(B, T, C, H, W)

In [ ]:
tensor_to_video_pane(videos, nrow=3, padding=1, zoom=4, format="mp4", cmap="magma")